In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import numpy as onp
import jax
from ase.visualize import view
from ase.atoms import Atoms

from msmjax.core.shortrange import make_eval_pair_pot, _gen_supercell
from msmjax.utils.general import (
    evaluate_structure_with_lammps_p3m,
    path_input_structures,
)

LAMMPS_EXECUTABLE = "/home/florian/Downloads/lammps-static/bin/lmp"

In [2]:
# TODO: Also compute reference results for stress? (LAMMPS compute pressure command?)
# TODO: Charge gradient (= electrostatic potential at particle positions)?

# Function definitions

In [3]:
def calc_energy_ref_nonperiodic(positions, charges):
    n_dim = positions.shape[1]
    compute_pair_term = make_eval_pair_pot(
        kernel_fn=lambda x: 1.0 / x, pbc=(False,) * n_dim
    )
    return compute_pair_term(positions, charges)


@jax.jit
def calc_reference_results_nonperiodic(positions, charges):
    value, grad = jax.value_and_grad(calc_energy_ref_nonperiodic, argnums=0)(
        positions, charges
    )
    return value, -grad

In [4]:
def load_one_structure(n_particles):
    structures = onp.load(
        path_input_structures / ("structures_" + str(n_particles) + ".npz")
    )
    # TODO: Also test different structures of the same number of particles?
    #  (i.e., other values for idx_structure than 0)
    idx_structure = 0
    pos = structures["positions"][idx_structure]
    chg = structures["charges"][idx_structure]
    cell = structures["cells"][idx_structure]
    pos = pos.astype(onp.float64)
    chg = chg.astype(onp.float64)
    cell = cell.astype(onp.float64)
    return pos, chg, cell

# Non-periodic

## Cubic

In [5]:
(pos, chg, cell) = load_one_structure(10000)

e_ref, f_ref = calc_reference_results_nonperiodic(pos, chg)

fname = "nonperiodic_cubic.npz"
onp.savez_compressed(
    fname, positions=pos, charges=chg, cell=cell, energy=e_ref, forces=f_ref
)

## Orthorhombic with different side lengths

The intention is that the side lengths are so different that the minimum number of grid points is reached along some axis before the others.

In [6]:
(pos, chg, cell) = load_one_structure(1500)
(pos, chg, cell) = _gen_supercell(pos, chg, cell, supercell_diag=(3, 2, 1))
scaled_pos = pos @ onp.linalg.pinv(cell)
axis_stretch_factors = onp.array([1.1, 1.0, 0.9])
cell *= axis_stretch_factors
pos = scaled_pos @ cell

e_ref, f_ref = calc_reference_results_nonperiodic(pos, chg)

fname = "nonperiodic_ortho-different-sidelengths.npz"
onp.savez_compressed(
    fname, positions=pos, charges=chg, cell=cell, energy=e_ref, forces=f_ref
)

atoms = Atoms(positions=pos, cell=cell, charges=chg)
view(atoms)

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

## Triclinic

In [7]:
(pos, chg, cell) = load_one_structure(10000)

atoms = Atoms(positions=pos, charges=chg, cell=cell)
new_lengths = onp.diag(cell) * (0.8, 1.0, 1.25)
new_angles = [75, 90, 120]
nonortho_cell = onp.concatenate([new_lengths, new_angles])
atoms.set_cell(nonortho_cell, scale_atoms=True)
pos = onp.array(atoms.get_positions())
chg = onp.array(atoms.get_initial_charges())
cell = onp.array(atoms.get_cell()[...])

e_ref, f_ref = calc_reference_results_nonperiodic(pos, chg)

fname = "nonperiodic_triclinic.npz"
onp.savez_compressed(
    fname, positions=pos, charges=chg, cell=cell, energy=e_ref, forces=f_ref
)

view(atoms)

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

# Periodic

## Cubic

In [8]:
(pos, chg, cell) = load_one_structure(500)

e_ref, f_ref = evaluate_structure_with_lammps_p3m(
    pos, chg, cell, LAMMPS_EXECUTABLE, max_neighbors_one_atom=10000
)

fname = "periodic_cubic.npz"
onp.savez_compressed(
    fname, positions=pos, charges=chg, cell=cell, energy=e_ref, forces=f_ref
)

## Orthorhombic with different side lengths

The intention is that the side lengths are so different that the grid is reduced to a single point along some axis faster than along the others.

In [9]:
(pos, chg, cell) = load_one_structure(500)
(pos, chg, cell) = _gen_supercell(pos, chg, cell, supercell_diag=(3, 2, 1))
scaled_pos = pos @ onp.linalg.pinv(cell)
axis_stretch_factors = onp.array([1.1, 1.0, 0.9])
cell *= axis_stretch_factors
pos = scaled_pos @ cell

e_ref, f_ref = evaluate_structure_with_lammps_p3m(
    pos, chg, cell, LAMMPS_EXECUTABLE, max_neighbors_one_atom=10000
)

fname = "periodic_ortho-different-sidelengths.npz"
onp.savez_compressed(
    fname, positions=pos, charges=chg, cell=cell, energy=e_ref, forces=f_ref
)

atoms = Atoms(positions=pos, cell=cell, charges=chg)
view(atoms)

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

## Triclinic

In [10]:
(pos, chg, cell) = load_one_structure(500)

atoms = Atoms(positions=pos, charges=chg, cell=cell)
new_lengths = onp.diag(cell) * (0.8, 1.0, 1.25)
new_angles = [75, 90, 120]
nonortho_cell = onp.concatenate([new_lengths, new_angles])
atoms.set_cell(nonortho_cell, scale_atoms=True)
pos = onp.array(atoms.get_positions())
chg = onp.array(atoms.get_initial_charges())
cell = onp.array(atoms.get_cell()[...])

e_ref, f_ref = evaluate_structure_with_lammps_p3m(
    pos, chg, cell, LAMMPS_EXECUTABLE, max_neighbors_one_atom=10000
)

fname = "periodic_triclinic.npz"
onp.savez_compressed(
    fname, positions=pos, charges=chg, cell=cell, energy=e_ref, forces=f_ref
)

view(atoms)

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>